In [14]:
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample

### Plot Styles

In [15]:
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120})

### Load Data Path

In [16]:
EXTRACTED_DIR = Path("../Extracted_data")          
LOG_CSV       = Path("../extraction_log.csv")        
CLEAN_CSV     = Path("../dataset_clean.csv")        # output: cleaned, original dist -> fairness/SHAP
OUTPUT_CSV    = Path("../dataset_balanced.csv")     # output: resampled -> DistilBERT training

### Fix UTF-8

In [17]:
import ftfy

# Apply ftfy to all .txt files before cleaning
fixed_count = 0
for txt_file in sorted(EXTRACTED_DIR.glob("**/*.txt")):
    raw   = txt_file.read_text(encoding="utf-8", errors="replace")
    fixed = ftfy.fix_text(raw)
    if fixed != raw:
        txt_file.write_text(fixed, encoding="utf-8")
        fixed_count += 1

print(f"ftfy applied (v)")
print(f"Files with encoding artifacts fixed: {fixed_count}")

# Quick sanity check — show before/after on a sample
sample = 'Responsbilityâ€¢ Maintain Â· clean Â§ safe'
print(f"\nBefore: {sample}")
print(f"After : {ftfy.fix_text(sample)}")


ftfy applied (v)
Files with encoding artifacts fixed: 0

Before: Responsbilityâ€¢ Maintain Â· clean Â§ safe
After : Responsbility• Maintain · clean § safe


### Text Cleaning & PII Removal

In [18]:
# PII patterns 
_PHONE   = re.compile(r"(?:\+?1[\s\-.])?" r"(?:\(?\d{3}\)?[\s\-.])" r"\d{3}[\s\-.]?\d{4}")
_EMAIL   = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
_URL     = re.compile(
    r"(?:https?://|www\.)"
    r"(?:linkedin\.com|github\.com|twitter\.com|facebook\.com|instagram\.com)"
    r"[^\s]*",
    re.IGNORECASE
)
_ADDRESS = re.compile(
    r"\d{1,5}\s+(?:[A-Z][a-z]+\s+){1,4}"
    r"(?:St|Ave|Rd|Blvd|Dr|Ln|Way|Court|Ct|Place|Pl|Suite|Ste)\.?",
    re.IGNORECASE
)
_NAME_TOP = re.compile(r"^(?:[A-Z][a-z]+\s+){1,3}[A-Z][a-z]+")
_PHOTO    = re.compile(r"(?:profile\s*photo|photo\.(?:jpg|jpeg|png|gif))", re.IGNORECASE)
_PAGE_TAG = re.compile(r"---\s*Page\s*\d+(?:\s*\(OCR\))?\s*---")

# Noise symbols (after ftfy, non-alphanumeric except kept chars) 
# Keep: letters, digits, whitespace, hyphen (compound words), period (abbreviations), comma, @ (for PII regex to catch email first)
_SYMBOLS  = re.compile(r"[^a-zA-Z0-9\s\-\.,@]")


def clean_resume(text: str) -> str:
    # 1. Remove page markers
    text = _PAGE_TAG.sub(" ", text)

    # 2. Strip PII
    text = _NAME_TOP.sub("[NAME]", text, count=1)
    text = _PHONE.sub("[PHONE]", text)
    text = _EMAIL.sub("[EMAIL]", text)
    text = _URL.sub("[SOCIAL]", text)
    text = _ADDRESS.sub("[ADDRESS]", text)
    text = _PHOTO.sub("[PHOTO]", text)

    # 3. Remove noise symbols (•, ·, §, –, €, ©, etc.)
    text = _SYMBOLS.sub(" ", text)

    # 4. Collapse whitespace (tabs, newlines -> single space)
    text = re.sub(r"[\t\r\n]+", " ", text)
    text = re.sub(r" {2,}", " ", text)

    # 5. Lowercase
    return text.lower().strip()

# Sanity check
print("\nSymbol removal test:")
test = "Responsibilities: â€¢ Maintain Â· clean Â§ safe – organized workspace"
test_fixed = ftfy.fix_text(test)
print(f"  raw   : {test}")
print(f"  ftfy  : {test_fixed}")
print(f"  clean : {clean_resume(test_fixed)}")

print("\nSample output (300 chars):")
sample_path = next(EXTRACTED_DIR.glob("**/*.txt"))
raw = sample_path.read_text(encoding="utf-8")
print(clean_resume(raw)[:300])



Symbol removal test:
  raw   : Responsibilities: â€¢ Maintain Â· clean Â§ safe – organized workspace
  ftfy  : Responsibilities: • Maintain · clean § safe – organized workspace
  clean : responsibilities maintain clean safe organized workspace

Sample output (300 chars):
accountant summary financial accountant specializing in financial planning, reporting and analysis within the department of defense. highlights account reconciliations results-oriented accounting operations professional financial reporting analysis of financial systems critical thinking erp enterpri


In [19]:
cleaning_report = []

for txt_file in sorted(EXTRACTED_DIR.glob("**/*.txt")):
    raw     = txt_file.read_text(encoding="utf-8")
    cleaned = clean_resume(raw)

    wc_before = len(raw.split())
    wc_after  = len(cleaned.split())

    txt_file.write_text(cleaned, encoding="utf-8")

    pii_tags = ["[phone]", "[email]", "[social]", "[address]", "[name]"]
    cleaning_report.append({
        "occupation": txt_file.parent.name,
        "filename":   txt_file.name,
        "wc_before":  wc_before,
        "wc_after":   wc_after,
        "pii_found":  any(tag in cleaned for tag in pii_tags),
    })

clean_report_df = pd.DataFrame(cleaning_report)
print(f"Files cleaned  : {len(clean_report_df)}")
print(f"PII detected   : {clean_report_df['pii_found'].sum()} file(s)")
print(f"Avg words before : {clean_report_df['wc_before'].mean():.0f}")
print(f"Avg words after  : {clean_report_df['wc_after'].mean():.0f}")


Files cleaned  : 2482
PII detected   : 0 file(s)
Avg words before : 812
Avg words after  : 812


In [20]:
# Reload df from cleaned .txt files
records = []
for occ_dir in sorted(EXTRACTED_DIR.iterdir()):
    if not occ_dir.is_dir():
        continue
    for txt_file in sorted(occ_dir.glob("*.txt")):
        text = txt_file.read_text(encoding="utf-8")
        records.append({
            "occupation": occ_dir.name,
            "filename":   txt_file.name,
            "text":       text,
            "char_count": len(text),
            "word_count": len(text.split()),
        })

df = pd.DataFrame(records)
df["tokens"]     = df["text"].apply(lambda t: re.findall(r"\b[a-z]{3,}\b", t))
df["vocab_size"] = df["tokens"].apply(lambda t: len(set(t)))

print(f"df reloaded: {len(df)} resumes")
print(df[["occupation", "filename", "word_count", "vocab_size"]].head(8))


df reloaded: 2482 resumes
   occupation      filename  word_count  vocab_size
0  ACCOUNTANT  10554236.txt        3500         916
1  ACCOUNTANT  10674770.txt        1040         413
2  ACCOUNTANT  11163645.txt         629         260
3  ACCOUNTANT  11759079.txt         853         367
4  ACCOUNTANT  12065211.txt         793         368
5  ACCOUNTANT  12202337.txt         753         314
6  ACCOUNTANT  12338274.txt         812         271
7  ACCOUNTANT  12442909.txt         641         315


In [21]:
# Save cleaned dataset (original distribution, no resampling)
# Used for fairness metrics & SHAP - reflects real-world class distribution
cols = ["occupation", "filename", "text", "word_count", "char_count", "vocab_size"]
df[cols].to_csv(CLEAN_CSV, index=False)

print(f"Saved : {CLEAN_CSV}")
print(f"Shape : {df.shape}")
print(f"\nClass distribution (original):")
print(df["occupation"].value_counts().to_string())
print("\n-> Used for fairness metrics & SHAP - reflects real-world class distribution")



Saved : ..\dataset_clean.csv
Shape : (2482, 7)

Class distribution (original):
occupation
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      119
ACCOUNTANT                118
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
FINANCE                   118
AVIATION                  117
FITNESS                   117
SALES                     116
BANKING                   115
CONSULTANT                115
HEALTHCARE                115
CONSTRUCTION              112
HR                        110
PUBLIC-RELATIONS          110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22

-> Used for fairness metrics & SHAP - reflects real-world class distribution


### Tokenizing & Encoding

Split at **resume level** first (stratified by occupation), then oversample only the train set, then tokenize each split separately.
This prevents data leakage where duplicate chunks from oversampling appear in both train and test.

```
dataset_clean.csv
      ↓  split by resume (70/15/15 stratified)
  train_raw → oversample minority → tokenize → train_tokenized.csv
  val_raw   → no resampling       → tokenize → val_tokenized.csv
  test_raw  → no resampling       → tokenize → test_tokenized.csv
```

In [22]:
from transformers import DistilBertTokenizerFast
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
print(f'Tokenizer loaded (v)')
print(f'Vocab size : {tokenizer.vocab_size:,}')
print(f'Max length : {tokenizer.model_max_length}')


Tokenizer loaded (v)
Vocab size : 30,522
Max length : 512


In [23]:
SEED    = 42
MAX_LEN = 512
STRIDE  = 256

# ── Step 1: Split at resume level (before any resampling) ─────────────────────
# Load clean dataset (one row per resume, original distribution)
df_clean = pd.read_csv(CLEAN_CSV)

train_raw, temp_raw = train_test_split(
    df_clean, test_size=0.30, stratify=df_clean['occupation'], random_state=SEED
)
val_raw, test_raw = train_test_split(
    temp_raw, test_size=0.50, stratify=temp_raw['occupation'], random_state=SEED
)

print('Split at resume level (no leakage):')
print(f'  Train : {len(train_raw):>5} resumes (70%)')
print(f'  Val   : {len(val_raw):>5} resumes (15%)')
print(f'  Test  : {len(test_raw):>5} resumes (15%)')

# ── Step 2: Oversample only train set to median count ─────────────────────────
TARGET_N   = int(train_raw['occupation'].value_counts().median())
print(f'\nOversampling train set to median: {TARGET_N} per class')

balanced_train = []
for occ in train_raw['occupation'].unique():
    occ_df = train_raw[train_raw['occupation'] == occ]
    n = len(occ_df)
    if n < TARGET_N:
        resampled = resample(occ_df, replace=True,  n_samples=TARGET_N, random_state=SEED)
        tag = f'oversampled  {n} → {TARGET_N}'
    elif n > TARGET_N:
        resampled = resample(occ_df, replace=False, n_samples=TARGET_N, random_state=SEED)
        tag = f'undersampled {n} → {TARGET_N}'
    else:
        resampled = occ_df
        tag = f'unchanged    {n}'
    balanced_train.append(resampled)
    print(f'  {occ:<25} {tag}')

train_balanced = pd.concat(balanced_train).reset_index(drop=True)
print(f'\nBalanced train: {len(train_balanced)} resumes')
print(f'Val  (untouched): {len(val_raw)} resumes')
print(f'Test (untouched): {len(test_raw)} resumes')


Split at resume level (no leakage):
  Train :  1737 resumes (70%)
  Val   :   372 resumes (15%)
  Test  :   373 resumes (15%)

Oversampling train set to median: 80 per class
  INFORMATION-TECHNOLOGY    undersampled 84 → 80
  CHEF                      undersampled 83 → 80
  CONSTRUCTION              oversampled  78 → 80
  AGRICULTURE               oversampled  44 → 80
  TEACHER                   oversampled  71 → 80
  BANKING                   unchanged    80
  ACCOUNTANT                undersampled 83 → 80
  AUTOMOBILE                oversampled  25 → 80
  APPAREL                   oversampled  68 → 80
  ENGINEERING               undersampled 83 → 80
  HEALTHCARE                unchanged    80
  ARTS                      oversampled  72 → 80
  SALES                     undersampled 81 → 80
  CONSULTANT                undersampled 81 → 80
  BUSINESS-DEVELOPMENT      undersampled 83 → 80
  ADVOCATE                  undersampled 83 → 80
  HR                        oversampled  77 → 80
  D

In [24]:
# ── Step 3: Tokenize each split separately ─────────────────────────────────────
def chunk_resume(text: str, label: str, filename: str) -> list:
    """
    Tokenizes a resume and splits into overlapping 512-token chunks (stride=256).
    Each chunk gets the same occupation label as the source resume.
    """
    encoding = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False,
        return_tensors=None,
    )
    all_ids = encoding['input_ids']

    CLS = tokenizer.cls_token_id
    SEP = tokenizer.sep_token_id
    effective_len = MAX_LEN - 2

    chunks, start, chunk_id = [], 0, 0
    while start < len(all_ids):
        end    = min(start + effective_len, len(all_ids))
        ids    = [CLS] + all_ids[start:end] + [SEP]
        pad_len = MAX_LEN - len(ids)

        chunks.append({
            'filename':       filename,
            'occupation':     label,
            'chunk_id':       chunk_id,
            'input_ids':      ids + [tokenizer.pad_token_id] * pad_len,
            'attention_mask': [1] * len(ids) + [0] * pad_len,
            'n_tokens':       len(ids),
        })
        if end == len(all_ids): break
        start += STRIDE
        chunk_id += 1

    return chunks


def tokenize_split(split_df: 'pd.DataFrame', split_name: str) -> 'pd.DataFrame':
    all_chunks = []
    for _, row in split_df.iterrows():
        all_chunks.extend(chunk_resume(row['text'], row['occupation'], row['filename']))
    result = pd.DataFrame(all_chunks)
    print(f'{split_name:<6}: {len(split_df):>5} resumes → {len(result):>6,} chunks '
          f'(avg {len(result)/len(split_df):.1f} chunks/resume)')
    return result


print('Tokenizing...')
df_train_tok = tokenize_split(train_balanced, 'Train')
df_val_tok   = tokenize_split(val_raw,        'Val')
df_test_tok  = tokenize_split(test_raw,       'Test')

# Sanity check — no filename overlap between splits
train_files = set(df_train_tok['filename'])
val_files   = set(df_val_tok['filename'])
test_files  = set(df_test_tok['filename'])
assert len(train_files & test_files) == 0, 'Leakage: train/test overlap!'
assert len(val_files   & test_files) == 0, 'Leakage: val/test overlap!'
print('\nLeakage check passed (v) — no filename overlap across splits')


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1288 > 512). Running this sequence through the model will result in indexing errors


Tokenizing...
Train :  1920 resumes →  6,614 chunks (avg 3.4 chunks/resume)
Val   :   372 resumes →  1,322 chunks (avg 3.6 chunks/resume)
Test  :   373 resumes →  1,258 chunks (avg 3.4 chunks/resume)

Leakage check passed (v) — no filename overlap across splits


In [25]:
# ── Step 4: Save all three splits ─────────────────────────────────────────────
import json as _json

TRAIN_TOK_CSV = Path('../train_tokenized.csv')
VAL_TOK_CSV   = Path('../val_tokenized.csv')
TEST_TOK_CSV  = Path('../test_tokenized.csv')

def save_tokenized(df: 'pd.DataFrame', path: 'Path'):
    out = df.copy()
    out['input_ids']      = out['input_ids'].apply(_json.dumps)
    out['attention_mask'] = out['attention_mask'].apply(_json.dumps)
    out[['filename', 'occupation', 'chunk_id',
         'input_ids', 'attention_mask', 'n_tokens']].to_csv(path, index=False)

save_tokenized(df_train_tok, TRAIN_TOK_CSV)
save_tokenized(df_val_tok,   VAL_TOK_CSV)
save_tokenized(df_test_tok,  TEST_TOK_CSV)

print('Saved:')
print(f'  train_tokenized.csv → {len(df_train_tok):,} chunks')
print(f'  val_tokenized.csv   → {len(df_val_tok):,} chunks')
print(f'  test_tokenized.csv  → {len(df_test_tok):,} chunks')
print()
print('Summary:')
print(f'  dataset_clean.csv      → {len(df_clean)} resumes  (original dist, for fairness/SHAP)')
print(f'  train_tokenized.csv    → {len(df_train_tok):,} chunks   (balanced, for training)')
print(f'  val_tokenized.csv      → {len(df_val_tok):,} chunks   (original dist, for validation)')
print(f'  test_tokenized.csv     → {len(df_test_tok):,} chunks   (original dist, for evaluation)')


Saved:
  train_tokenized.csv → 6,614 chunks
  val_tokenized.csv   → 1,322 chunks
  test_tokenized.csv  → 1,258 chunks

Summary:
  dataset_clean.csv      → 2482 resumes  (original dist, for fairness/SHAP)
  train_tokenized.csv    → 6,614 chunks   (balanced, for training)
  val_tokenized.csv      → 1,322 chunks   (original dist, for validation)
  test_tokenized.csv     → 1,258 chunks   (original dist, for evaluation)
